### Imports

In [ ]:
import requests, re, json
import pandas as pd
headers = {"User-Agent": "Mozilla/5.0 (course-demo; personal use)"}

### Check for robots

In [ ]:
for base in ["https://www.rimi.lt", "https://barbora.lt"]:
    r = requests.get(base + "/robots.txt", headers=headers, timeout=20)
    print("=====", base, r.status_code)
    print(r.text[:1500])

### Check Rimi

In [ ]:
def diagnose(url, needle):
    r = requests.get(url, headers=headers, timeout=20)
    html = r.text
    print("status:", r.status_code, "| size:", len(html))
    print("needle count:", html.count(needle))
    print("window.* variables:", re.findall(r"window\.\w+\s*=", html))
    print("ld+json blocks:", html.count("application/ld+json"))
    print("__NEXT_DATA__:", "__NEXT_DATA__" in html)
    print("data-* attrs with 'price':", len(re.findall(r'data-[\w-]*price[\w-]*=', html, re.I)))
    return html

rimi_url = "https://www.rimi.lt/e-parduotuve/lt/produktai/mesa-ir-zuvis-/sviezia-mesa/c/SH-9-8"
rimi_html = diagnose(rimi_url, "antrekot")

In [ ]:
def show_context(html, needle, width=400, max_hits=2):
    start = 0
    for n in range(max_hits):
        i = html.find(needle, start)
        if i == -1:
            break
        print(f"--- hit {n+1} at position {i} ---")
        print(html[max(0, i-width): i+width].replace("\n", " "))
        print()
        start = i + 1500   # jump ahead so hit 2 is a different place

show_context(rimi_html, "antrekot")

print("'/produktai/' count:", rimi_html.count("/produktai/"))
print("'data-gtm' count:", rimi_html.count("data-gtm"))
print("'<li' count:", rimi_html.count("<li"))

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(rimi_html, "html.parser")

# every product card on the page
cards = soup.select("li.product-grid__item")
print("cards:", len(cards))

# the card's JSON attribute -> Python dict (BeautifulSoup handles escaping for us)
card = cards[0]
div = card.select_one("div[data-gtm-eec-product]")
print(json.loads(div["data-gtm-eec-product"]))

# 1) all the visible text in the card, which should include the unit price
print(card.get_text(" ", strip=True))

# 2) the card's link and image
a = card.select_one("a.card__url")
print("link:", a["href"] if a else None)
img = card.select_one("img")
print("image:", img.get("src") if img else None)

In [ ]:
from bs4 import BeautifulSoup
import json, re

def parse_rimi_page(html):
    """Return a list of normalized product dicts from one Rimi category page."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # each product is one <li class="product-grid__item">
    for card in soup.select("li.product-grid__item"):

        # the analytics JSON gives us id, name and pack price
        div = card.select_one("div[data-gtm-eec-product]")
        if div is None:          # skip anything that isn't a real product card
            continue
        info = json.loads(div["data-gtm-eec-product"])

        # unit price is only in the visible text, e.g. "Kaina už vienetą: 39,95 €/kg"
        text = card.get_text(" ", strip=True)
        m = re.search(r"Kaina už vienetą:\s*(\d+(?:[ \d]*)?,\d+)\s*€/(\w+)", text)
        unit_price = float(m.group(1).replace(" ", "").replace(",", ".")) if m else None
        unit = m.group(2) if m else None

        # link and image
        a = card.select_one("a.card__url")
        href = a["href"] if a else None
        img = card.select_one("img")

        # category = the readable part of the link between "produktai/" and "/p/"
        category = None
        if href and "/produktai/" in href:
            category = href.split("/produktai/")[1].split("/p/")[0]

        rows.append({
            "store": "rimi",
            "id": info["id"],
            "name": info["name"],
            "brand": "RIMI" if "RIMI" in info["name"] else info.get("brand"),  # rough for now
            "price": info["price"],
            "unit_price": unit_price,
            "unit": unit,
            "category": category,
            "image": img.get("src") if img else None,
            "url": "https://www.rimi.lt" + href if href else None,
            "on_promo": None,   # TODO: we haven't looked at how Rimi shows promotions
        })
    return rows

rimi_rows = parse_rimi_page(rimi_html)
rimi_df = pd.DataFrame(rimi_rows)
print(len(rimi_df))
print(rimi_df[["name", "price", "unit_price", "unit"]].head(10))
print("missing unit_price:", rimi_df["unit_price"].isna().sum())

In [ ]:
import time
def scrape_rimi(category_code="SH-9", slug="mesa-ir-zuvis-", max_pages=3, delay=2.0):
    # base address of the category page (same for every page number)
    base_url = f"https://www.rimi.lt/e-parduotuve/lt/produktai/{slug}/c/{category_code}"
    rows, seen_ids = [], set()

    for page in range(1, max_pages + 1):
        # requests builds "?currentPage=2&pageSize=40&query=..." for us
        # and encodes special characters like ":" as %3A
        params = {
            "currentPage": page,
            "pageSize": 40,
            "query": f":relevance:allCategories:{category_code}:assortmentStatus:inAssortment",
        }
        r = requests.get(base_url, params=params, headers=headers, timeout=20)
        r.raise_for_status()

        page_rows = parse_rimi_page(r.text)

        # keep only products we haven't seen yet. Some shops repeat the last
        # page when you ask for a page that doesn't exist, so a page with
        # zero NEW products means we've reached the end.
        new_rows = [row for row in page_rows if row["id"] not in seen_ids]
        print(f"page {page}: {len(page_rows)} cards, {len(new_rows)} new")
        if not new_rows:
            break

        seen_ids.update(row["id"] for row in new_rows)
        rows += new_rows
        time.sleep(delay)   # one request every 2 seconds

    return rows

rimi_rows = scrape_rimi(max_pages=3)
rimi_df = pd.DataFrame(rimi_rows)
print(len(rimi_df), "products,", rimi_df["unit_price"].isna().sum(), "missing unit_price")

rimi_df.to_json("data/raw/rimi_meat.json", orient="records", force_ascii=False, indent=1)